# DMart Analysis — All Query Questions in Python

This notebook contains the Pandas/Python equivalents of all query-based analysis questions from the uploaded DMart Analysis notebook.

 1. Load all datasets

In [1]:
import pandas as pd

customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
stores = pd.read_csv("stores.csv")
suppliers = pd.read_csv("suppliers.csv")
purchase_orders = pd.read_csv("purchase_orders.csv")
inventory = pd.read_csv("inventory.csv")
sales = pd.read_csv("sales.csv")
categories = pd.read_csv("categories.csv")

Q1. Total Sales Per Store

In [2]:
df = sales.groupby("Store_ID", as_index=False)["Sales_Amount"].sum()
df = df.rename(columns={"Sales_Amount": "Total_Sales"})
df

,Store_ID,Total_Sales
0,ST01,49047926.92
1,ST02,50328840.39
2,ST03,29398037.03
3,ST04,12341787.10
4,ST05,15418004.57
5,ST06,19564992.00
6,ST07,24319085.60
7,ST08,17670032.10
8,ST09,21717811.84
9,ST10,26236106.18


 Q2. Top 10 Best Selling Products

In [3]:
df = sales.merge(
    products[["Product_ID", "Product_Name"]],
    on="Product_ID",
    how="inner"
)

df = df.groupby(
    ["Product_ID", "Product_Name"],
    as_index=False
).agg(
    Total_Qty_Sold=("Quantity", "sum"),
    Total_Sales=("Sales_Amount", "sum")
)

df = df.sort_values("Total_Qty_Sold", ascending=False).head(10)
df

,Product_ID,Product_Name,Total_Qty_Sold,Total_Sales
136,P0137,Tata Tea Tea Powder 250g - Variant 4,5456,2307049.98
116,P0117,Nescafe Instant Coffee 50g - Variant 2,5412,85174.36
101,P0102,Bisleri Soda Water 600ml,5200,2115354.99
118,P0119,Pepsi Lemon Drink 600ml - Variant 2,4826,2030197.00
120,P0121,Fanta Cold Drink 1.25L - Variant 3,4728,1565970.52
119,P0120,Thums Up Cold Drink 750ml - Variant 3,4709,721115.08
124,P0125,Pepsi Energy Drink 250ml - Variant 3,4655,1683012.75
132,P0133,Tropicana Packaged Juice 1L - Variant 4,4587,925809.01
131,P0132,Fanta Cold Drink 1.25L - Variant 4,4583,1119656.69
117,P0118,Red Label Green Tea 100g - Variant 2,4572,954116.10


 Q3. Monthly Sales Trend

In [4]:
sales["Transaction_Date"] = pd.to_datetime(sales["Transaction_Date"])
sales["Year"] = sales["Transaction_Date"].dt.year
sales["Month"] = sales["Transaction_Date"].dt.month

df = sales.groupby(
    ["Year", "Month"],
    as_index=False
)["Sales_Amount"].sum()

df = df.rename(columns={"Sales_Amount": "Total_Sales_Per_Month"})
df = df.sort_values(["Year", "Month"])
df

,Year,Month,Total_Sales_Per_Month
0,2023,1,8114860.06
1,2023,2,7043145.27
2,2023,3,9410712.46
3,2023,4,8198619.79
4,2023,5,8658662.34
5,2023,6,7564441.50
6,2023,7,8602559.41
7,2023,8,8357572.11
8,2023,9,8706492.66
9,2023,10,7897779.71


 Q4. Top 10 Customers by Total Spending

In [5]:
df = sales.merge(
    customers[["Customer_ID", "Customer_Name"]],
    on="Customer_ID",
    how="inner"
)

df = df.groupby(
    ["Customer_ID", "Customer_Name"],
    as_index=False
)["Sales_Amount"].sum()

df = df.rename(columns={"Sales_Amount": "Total_Spend"})
df["Total_Spend"] = df["Total_Spend"].round(2)
df = df.sort_values("Total_Spend", ascending=False).head(10)
df

,Customer_ID,Customer_Name,Total_Spend
12034,C12095,Neel Dani,516119.56
1942,C01956,Yug Singhal,502435.93
1583,C01596,Imaran Gala,456097.93
6758,C06798,Amrita Oak,447422.10
12309,C12372,Ekaja Bains,438090.61
8710,C08755,Omaja Virk,437982.43
1321,C01334,Faraj Reddy,431153.75
12128,C12189,Qabil Kamdar,409496.70
8829,C08874,Reva Buch,400715.35
4242,C04271,Frederick Majumdar,391753.40


 Q5. Find Repeated Customers

In [6]:
df = sales.groupby("Customer_ID").size().reset_index(name="Customer_Ordered")
df = df[df["Customer_Ordered"] > 1]
df

,Customer_ID,Customer_Ordered
0,C00001,14
1,C00002,10
2,C00003,10
3,C00004,5
4,C00005,25
...,...,...
14925,C14996,6
14926,C14997,24
14927,C14998,17
14928,C14999,9


In [7]:
# Optional: include customer names
df = df.merge(
    customers[["Customer_ID", "Customer_Name"]],
    on="Customer_ID",
    how="left"
)
df

,Customer_ID,Customer_Ordered,Customer_Name
0,C00001,14,Chakrika Badal
1,C00002,10,Udyati Raghavan
2,C00003,10,Dhruv Handa
3,C00004,5,Tristan Atwal
4,C00005,25,Utkarsh Bakshi
...,...,...,...
14728,C14996,6,Shivani Bhandari
14729,C14997,24,Jason Sawhney
14730,C14998,17,Jagrati Agrawal
14731,C14999,9,Rajeshri Bali


 Q6. Which Store Performs Best Every Month?

In [8]:
sales["Transaction_Date"] = pd.to_datetime(sales["Transaction_Date"])
sales["Year"] = sales["Transaction_Date"].dt.year
sales["Month"] = sales["Transaction_Date"].dt.month

monthly_sales = sales.groupby(
    ["Year", "Month", "Store_ID"],
    as_index=False
)["Sales_Amount"].sum()

monthly_sales = monthly_sales.rename(
    columns={"Sales_Amount": "Total_Sales"}
)

monthly_sales["Rank"] = monthly_sales.groupby(
    ["Year", "Month"]
)["Total_Sales"].rank(
    method="dense",
    ascending=False
)

df = monthly_sales[monthly_sales["Rank"] == 1]
df = df.sort_values(["Year", "Month"])
df

,Year,Month,Store_ID,Total_Sales,Rank
0,2023,1,ST01,1372354.87,1.0
13,2023,2,ST02,1181999.98,1.0
25,2023,3,ST02,1604273.36,1.0
36,2023,4,ST01,1390362.27,1.0
48,2023,5,ST01,1313785.72,1.0
61,2023,6,ST02,1140638.59,1.0
73,2023,7,ST02,1396371.28,1.0
84,2023,8,ST01,1421943.41,1.0
97,2023,9,ST02,1620944.19,1.0
108,2023,10,ST01,1264738.31,1.0


 Q7. Average Daily Sales Per Store

In [9]:
sales["Transaction_Date"] = pd.to_datetime(sales["Transaction_Date"])

daily_sales = sales.groupby(
    ["Store_ID", "Transaction_Date"],
    as_index=False
)["Sales_Amount"].sum()

daily_sales = daily_sales.rename(
    columns={"Sales_Amount": "Daily_Sales"}
)

df = daily_sales.groupby(
    "Store_ID",
    as_index=False
)["Daily_Sales"].mean()

df = df.rename(columns={"Daily_Sales": "Avg_Daily_Sales"})
df["Avg_Daily_Sales"] = df["Avg_Daily_Sales"].round(2)
df = df.sort_values("Avg_Daily_Sales", ascending=False)
df

,Store_ID,Avg_Daily_Sales
1,ST02,45920.47
0,ST01,44751.76
11,ST12,28848.17
2,ST03,26823.03
9,ST10,23938.05
6,ST07,22188.95
8,ST09,19815.52
5,ST06,17851.27
7,ST08,16122.29
4,ST05,14067.52


 Q8. Which Category Generated the Highest Revenue in Each Store?

The original notebook's SQL for this section actually calculates category-level revenue, not store-wise category revenue. The code below follows that original query logic.

In [10]:
df = sales.merge(
    products[["Product_ID", "Category_ID"]],
    on="Product_ID",
    how="inner"
)

df = df.merge(
    categories[["Category_ID", "Category_Name"]],
    on="Category_ID",
    how="inner"
)

df = df.groupby(
    ["Category_ID", "Category_Name"],
    as_index=False
)["Sales_Amount"].sum()

df = df.rename(columns={"Sales_Amount": "Revenue"})
df = df.sort_values("Revenue")
df

,Category_ID,Category_Name,Revenue
12,CAT13,Fruits,2796918.90
13,CAT14,Vegetables,3409060.75
19,CAT20,Pet Care,3453082.77
14,CAT15,Stationery,4820482.21
4,CAT05,Bakery,5480889.13
8,CAT09,Cleaning Supplies,5903542.02
1,CAT02,Dairy,10159627.18
9,CAT10,Kitchen Essentials,11004558.40
17,CAT18,Home Decor,11128742.61
10,CAT11,Baby Care,13155876.90


 Q9. Which Category Generates Maximum Revenue?

In [11]:
df = sales.merge(
    products[["Product_ID", "Category_ID"]],
    on="Product_ID",
    how="inner"
)

df = df.merge(
    categories[["Category_ID", "Category_Name"]],
    on="Category_ID",
    how="inner"
)

df = df.groupby(
    ["Category_ID", "Category_Name"],
    as_index=False
)["Sales_Amount"].sum()

df = df.rename(columns={"Sales_Amount": "Revenue"})
df["Revenue"] = df["Revenue"].round(2)
df = df.sort_values("Revenue", ascending=False)
df

,Category_ID,Category_Name,Revenue
0,CAT01,Groceries,41960439.22
2,CAT03,Beverages,38812154.98
18,CAT19,Electronics,32328772.62
3,CAT04,Snacks,31256797.25
6,CAT07,Personal Care,18309611.98
11,CAT12,Health Care,17393570.99
15,CAT16,Fashion,16793300.77
7,CAT08,Home Care,14864052.07
16,CAT17,Footwear,14572808.25
5,CAT06,Frozen Foods,13846750.96


In [12]:
# Only the maximum-revenue category
df.head(1)

,Category_ID,Category_Name,Revenue
0,CAT01,Groceries,41960439.22


Q10. Identify Products That Have Never Been Sold

In [14]:
df = products[
    ~products["Product_ID"].isin(sales["Product_ID"])
]

df

## Every product has been sold 

,Product_ID,Product_Name,Brand,Category_ID,Unit_Cost,Selling_Price,Profit_Margin_Percent,Launch_Date
